# RP-012 Official T2ICount warm start

Does one epoch of RP-010B Rich Prompt supervision improve class-name counting
when starting from the fully trained official checkpoint? The frozen reference
and the warm-start arm use identical initial model weights and the same
FSC147 validation evaluator. The arm uses the fixed 1,000 anchors and full
training donor pool, seed 3407, fresh AdamW and no scheduler. HNW and
density consistency are disabled. One epoch is exactly 1,000 updates.

Select **A100 40 GB** with high host RAM in Colab. An **L4 24 GB** is usable
if it passes the measured memory gate. Run cells in order. Keep
**RUN_TRAINING = False** until you review the frozen validation and GPU
preflight results. The warm start meets the validation decision rule if
MAE decreases and RMSE does not increase. Signed bias is prediction minus GT.


## Drive inputs and outputs

Upload this notebook and its matching source ZIP. Reuse the official
checkpoint, prompt bank, and pretrained/FSC147 assets already on Drive.

```text
MyDrive/
  RP012_upload/
    RP012_Colab.ipynb
    RP012_source.zip
  T2ICount-assets.zip
  T2ICount-assets/
    checkpoints/official/best_model_paper.pth
    prompts/fsc147_train1000_seed3407_gemma4_v3.json
    outputs/rp012/seed3407/run_v1/   # new outputs created by these cells
```

The existing asset ZIP contains `pretrained/` and `datasets/FSC147/`
with an optional `T2ICount-assets/` prefix. Expanded Drive directories
at the same paths under `T2ICount-assets/` also work if the ZIP is absent.
Keep at least 5 GiB free on Drive for the new checkpoint and reports.
Choose a new run ID for any retry so that prior outputs are preserved.


In [ ]:
from pathlib import Path
import hashlib, importlib.metadata, json, os, shlex, shutil, subprocess, sys, zipfile
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = Path('/content/drive/MyDrive/T2ICount-assets')
SOURCE_ZIP = Path('/content/drive/MyDrive/RP012_upload/RP012_source.zip')
ASSET_ZIP = Path('/content/drive/MyDrive/T2ICount-assets.zip')
REPO = Path('/content/T2ICount-RP012')
ASSET_ROOT = Path('/content/T2ICount-assets')
DATA_ROOT = ASSET_ROOT / 'datasets/FSC147'
CHECKPOINT = DRIVE_ROOT / 'checkpoints/official/best_model_paper.pth'
PROMPT_BANK = DRIVE_ROOT / 'prompts/fsc147_train1000_seed3407_gemma4_v3.json'
RUN_ID = 'run_v1'  # choose a new ID after an interrupted or failed attempt
assert RUN_ID and all(c.isalnum() or c in '_-' for c in RUN_ID)
OUTPUT_DIR = DRIVE_ROOT / 'outputs/rp012/seed3407' / RUN_ID
assert not OUTPUT_DIR.exists(), f'Output exists: {OUTPUT_DIR}; preserve it and choose a new RUN_ID'
NOTEBOOK_LOGS = OUTPUT_DIR / 'notebook'
NOTEBOOK_LOGS.mkdir(parents=True, exist_ok=False)
PREFLIGHT_PASSED = False

def stream(command, log_name):
    command = list(map(str, command))
    print('+', shlex.join(command), flush=True)
    with (NOTEBOOK_LOGS / log_name).open('a', encoding='utf-8') as log:
        process = subprocess.Popen(command, cwd=REPO, stdout=subprocess.PIPE,
                                   stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in process.stdout:
            print(line, end='', flush=True)
            log.write(line)
            log.flush()
        if process.wait() != 0:
            raise RuntimeError(f'Command failed ({process.returncode}); inspect {log.name}')

print('Official checkpoint:', CHECKPOINT)
print('New RP-012 outputs:', OUTPUT_DIR)


## Verify and unpack the source ZIP

The ZIP contains the current runner and shared RP-010/RP-011 helpers.
Its contents are checked against the notebook's manifest hash. The
provenance guard uses a scratch Git snapshot in the Colab runtime.


In [ ]:
EXPECTED_SOURCE_MANIFEST_SHA256 = 'e596083516e7c835a5f8c196e0c8d69b95165971c0d3f85723364a4f404bc6e1'
assert SOURCE_ZIP.is_file(), SOURCE_ZIP
assert not REPO.exists(), f'{REPO} already exists; use a fresh Colab runtime'
with zipfile.ZipFile(SOURCE_ZIP) as archive:
    raw = archive.read('SOURCE_MANIFEST.json')
    assert hashlib.sha256(raw).hexdigest() == EXPECTED_SOURCE_MANIFEST_SHA256, 'Source ZIP manifest mismatch'
    source_manifest = json.loads(raw)
    assert source_manifest['schema'] == 'rp012-colab-upload-v1'
    assert set(archive.namelist()) == set(source_manifest['files']) | {'SOURCE_MANIFEST.json'}
    REPO.mkdir(parents=True, exist_ok=False)
    for name, expected in source_manifest['files'].items():
        target = (REPO / name).resolve()
        assert REPO.resolve() in target.parents, f'Unsafe source member: {name}'
        data = archive.read(name)
        assert hashlib.sha256(data).hexdigest() == expected, f'Source changed: {name}'
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_bytes(data)
    (REPO / 'SOURCE_MANIFEST.json').write_bytes(raw)
for name in ('train.py', 'utils/regression_trainer.py', 'tools/train_rp012.py',
             'tools/train_rp010.py', 'tools/train_rp011.py', 'configs/rp012.json', 'configs/rp010b.json'):
    assert (REPO / name).is_file(), name
stream(['git', 'init', '-q'], 'source.log')
stream(['git', '-c', 'core.autocrlf=false', 'add', '.'], 'source.log')
stream(['git', '-c', 'user.name=RP012 upload snapshot', '-c', 'user.email=rp012@localhost',
        'commit', '-qm', 'RP012 uploaded working-tree source snapshot'], 'source.log')
print('Original source HEAD:', source_manifest['origin_head'])
print('Current uploaded file hashes verified:', len(source_manifest['files']))


## Install compatible dependencies

Reuse the project's Colab requirements and pinned CLIP/taming revisions.
The existing Torch, Torchvision and core numerical packages stay at their
installed versions. A fresh process checks the runner imports.


In [ ]:
constraints = Path('/content/rp012-runtime-constraints.txt')
protected = ('torch', 'torchvision', 'torchaudio', 'numpy', 'Pillow', 'h5py',
             'opencv-python', 'opencv-python-headless')
installed = {}
for package in protected:
    try:
        installed[package] = importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError:
        pass
assert 'torch' in installed and 'torchvision' in installed, 'Select a standard Colab GPU runtime'
constraints.write_text(''.join(f'{name}=={version}\n' for name, version in installed.items()))
stream([sys.executable, '-m', 'pip', 'install', '--upgrade-strategy', 'only-if-needed',
        '-c', constraints, '-r', REPO / 'requirements-colab.txt'], 'dependencies.log')
for package, version in installed.items():
    assert importlib.metadata.version(package) == version, f'Protected package changed: {package}'
taming = REPO / 'src/taming-transformers'
assert taming.is_dir(), 'Pinned editable taming-transformers checkout missing'
os.environ['PYTHONPATH'] = os.pathsep.join([str(REPO), str(taming), os.environ.get('PYTHONPATH', '')])
stream([sys.executable, '-c', 'import torch, torchvision, transformers, taming; '
        'from tools.train_rp012 import plan; print("RP-012 imports OK")'], 'dependencies.log')
freeze = subprocess.check_output([sys.executable, '-m', 'pip', 'freeze'], cwd=REPO, text=True)
(NOTEBOOK_LOGS / 'pip-freeze.txt').write_text(freeze)
shutil.copyfile(constraints, NOTEBOOK_LOGS / constraints.name)


## Check initial capacity

Start with at least 21 GiB free VRAM and 8 GiB available host RAM.
The GPU smoke records peak allocated and reserved memory and estimates
training headroom including AdamW moments. Precision stays FP32.


In [ ]:
gpu_probe = r'''
import json, torch
from pathlib import Path
from tools.train_rp010 import host_free_bytes, runtime_signature
assert torch.cuda.is_available(), 'CUDA unavailable: choose a GPU runtime'
assert torch.cuda.device_count() == 1, 'Use exactly one GPU'
free, total = torch.cuda.mem_get_info()
ram = host_free_bytes()
report = {'gpu': torch.cuda.get_device_name(0), 'free_bytes': free, 'total_bytes': total,
          'host_free_bytes': ram, 'runtime': runtime_signature()}
Path(__REPORT__).write_text(json.dumps(report, indent=2))
print(json.dumps(report, indent=2))
assert total >= 22 * 2**30, 'Need L4 24GB / A100 40GB class VRAM'
assert free >= 21 * 2**30, 'Insufficient free VRAM; do not change the experiment to fit'
assert ram is None or ram >= 8 * 2**30, 'Need at least 8 GiB free host RAM'
'''
gpu_probe = '\n'.join(line[4:] if line.startswith('    ') else line for line in gpu_probe.splitlines())
gpu_probe = gpu_probe.replace('__REPORT__', repr(str(NOTEBOOK_LOGS / 'gpu_runtime.json')))
stream([sys.executable, '-u', '-c', gpu_probe], 'gpu.log')


## Prepare FSC147 and pretrained assets

Extract only the existing pretrained and FSC147 subtrees to the Colab SSD.
The official checkpoint and RP-010B prompt bank are read from Drive.


In [ ]:
required = [ASSET_ROOT / 'pretrained/sd-v1-5/v1-5-pruned-emaonly.ckpt',
            ASSET_ROOT / 'pretrained/clip-vit-large-patch14/config.json',
            DATA_ROOT / 'FSC_147/Train_Test_Val_FSC_147.json']
if not all(path.is_file() for path in required):
    ASSET_ROOT.mkdir(parents=True, exist_ok=True)
    if ASSET_ZIP.is_file():
        with zipfile.ZipFile(ASSET_ZIP) as archive:
            selected = []
            for info in archive.infolist():
                name = info.filename.replace('\\', '/')
                if name.startswith('T2ICount-assets/'):
                    name = name[len('T2ICount-assets/'):]
                if not name.startswith(('pretrained/', 'datasets/FSC147/')) or info.is_dir():
                    continue
                target = (ASSET_ROOT / name).resolve()
                assert ASSET_ROOT.resolve() in target.parents, f'Unsafe asset member: {name}'
                selected.append((info, target))
            assert selected, 'ZIP must contain pretrained/ and datasets/FSC147/'
            needed = sum(info.file_size for info, target in selected if not target.is_file())
            assert shutil.disk_usage('/content').free > needed + 3 * 2**30, 'Insufficient SSD space'
            for index, (info, target) in enumerate(selected):
                if target.is_file():
                    assert target.stat().st_size == info.file_size, f'Incomplete local asset: {target}; use fresh runtime'
                    continue
                target.parent.mkdir(parents=True, exist_ok=True)
                with archive.open(info) as source, target.open('xb') as destination:
                    shutil.copyfileobj(source, destination, length=8 * 1024**2)
                if index % 500 == 0:
                    print(f'Extracted {index + 1}/{len(selected)} input files', flush=True)
    else:
        for relative in ('pretrained', 'datasets/FSC147'):
            source, target = DRIVE_ROOT / relative, ASSET_ROOT / relative
            assert source.is_dir(), f'Missing {ASSET_ZIP} and expanded input directory {source}'
            assert not target.exists(), f'Partial local inputs at {target}; use a fresh runtime'
            shutil.copytree(source, target)
for path in required + [CHECKPOINT, PROMPT_BANK]:
    assert path.is_file(), f'Missing required input: {path}'
print('Local FSC147:', DATA_ROOT)
print('Read-only warm start:', CHECKPOINT)


## Verify the experiment inputs

Check the official checkpoint SHA-256, the RP-010B source configuration,
prompt bank, ordered anchor selection, FSC147 split/class/annotation
hashes, and donor and validation coverage. The effective configuration
is printed and logged. Strict model loading follows in preflight.


In [ ]:
RUNNER = REPO / 'tools/train_rp012.py'
COMMON = ['--asset-root', str(ASSET_ROOT), '--checkpoint', str(CHECKPOINT),
          '--output-dir', str(OUTPUT_DIR), '--data-dir', str(DATA_ROOT),
          '--prompt-bank', str(PROMPT_BANK)]
stream([sys.executable, '-u', RUNNER, 'verify', *COMMON], 'input_validation.log')


## Run the frozen baseline and GPU preflight

The runner strictly loads the official model and evaluates all 1,286
validation images with class-name prompts using the RP-010 evaluator.
Both MAE and RMSE must be within 10% of the published 13.78 and 58.78.
This is a plausibility check. [Published T2ICount results](https://github.com/cha15yq/T2ICount#results).

The same model weights are then loaded for one three-forward/backward
Rich Prompt smoke. The preflight must record finite loss, finite nonzero
gradients, enough estimated free GPU memory for training, and zero
optimizer updates. A failure leaves the training cell gated.


In [ ]:
PREFLIGHT_PASSED = False
stream([sys.executable, '-u', RUNNER, 'preflight', *COMMON], 'preflight.log')
preflight_report = json.loads((OUTPUT_DIR / 'preflight/report.json').read_text())
assert preflight_report['status'] == 'passed', preflight_report.get('error')
official = preflight_report['baseline']
smoke = preflight_report['smoke']
assert official['strict_model_load'] and smoke['strict_model_load']
assert official['starting_weights_sha256'] == smoke['starting_weights_sha256']
assert official['optimizer_steps'] == smoke['optimizer_steps'] == 0
assert preflight_report['baseline_plausibility']['passed']
PREFLIGHT_PASSED = True
print('Frozen validation', official['metrics'])
print('GPU loss and gradient L2', smoke['loss'], smoke['gradient_l2'])
for key in ('peak_allocated_bytes', 'peak_reserved_bytes', 'training_estimate_bytes'):
    print(key.removesuffix('_bytes'), round(smoke[key] / 2**30, 2), 'GiB')
print('Preflight passed. Review the report before enabling training.')


## Optional one-epoch warm start

Enable this cell only after reviewing a successful preflight in this
session. The runner checks that source, input and runtime identities
still match, then strictly loads the identical starting weights into a
fresh trainer. It performs one epoch and saves the resulting checkpoint,
logs, validation predictions, and paired metrics to Drive.


In [ ]:
RUN_TRAINING = False
if RUN_TRAINING:
    assert PREFLIGHT_PASSED, 'Complete and review preflight first'
    stream([sys.executable, '-u', RUNNER, 'train', *COMMON, '--allow-training'],
           'warm_start_console.log')
else:
    print('Training is disabled.')


## Read the paired validation report

The report gives warm-start minus official differences for MAE, RMSE
and signed bias, and counts per-image improvements, worsenings and ties
based on absolute error. All pairs match by image ID, class and GT count.
A validation decision uses this one prespecified epoch.


In [ ]:
summary_path = OUTPUT_DIR / 'paired_validation.json'
if summary_path.is_file():
    print(summary_path.read_text())
    print('Per-image comparison', OUTPUT_DIR / 'paired_validation.csv')
    print('Trained checkpoint', OUTPUT_DIR / 'warm_start/best.pth')
else:
    print('Paired validation is available after the gated epoch.')
print('Preflight report', OUTPUT_DIR / 'preflight/report.json')
